# SQL-only companion notebook — YouTube category strategy queries

This notebook is the SQL companion to the main final project. It uses the same relational tables as the Python/write-up notebook, but here the SQL logic is made explicit and easy to audit.

The workflow is organized as one coherent analytical story instead of a loose query dump:
1. setup and schema checks,
2. data validation and sanity checks,
3. descriptive summaries,
4. join-based analysis across tables,
5. grouped comparisons,
6. window-function analysis,
7. subquery-based benchmarking,
8. a final decision-oriented scorecard.

Every query includes a comment block in the exact **what / how / why / expected output** format so the grader can quickly see both the technical logic and the analytical purpose. The outputs are designed to feed the final write-up, especially the sections on category performance, within-category leaders, engagement differences, concentration, and audience response quality.

## Setup and secure configuration

The notebook reads the API key safely, rebuilds the SQLite database if needed, and then runs a sequence of clearly documented SQL queries. The point of this companion file is not just to show syntax. It is to demonstrate a clean analytical workflow in which every query answers a business-relevant question that can be referenced in the final Python/write-up notebook.

In [1]:
import os
import re
import math
import warnings
import sqlite3
from pathlib import Path
from getpass import getpass

import numpy as np
import pandas as pd
import requests
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

DATA_DIR = Path("youtube_final_cache")
DATA_DIR.mkdir(exist_ok=True)
DB_PATH = DATA_DIR / "youtube_final_project.sqlite"

API_KEY = os.environ.get("YOUTUBE_API_KEY")
if not API_KEY:
    API_KEY = getpass("Enter YouTube Data API key (input hidden): ").strip()

if not API_KEY:
    raise ValueError("A YouTube Data API key is required. Set YOUTUBE_API_KEY or enter the key when prompted.")

SEED_CHANNELS = [
    "@MKBHD",
    "@TED",
    "@NBA",
    "@WIRED",
    "@NPRMusic",
    "@FoxNews",
]

REGION_CODE = "US"
MAX_UPLOADS_PER_CHANNEL = 250
MAX_VIDEOS_PER_CATEGORY = 60
FETCH_COMMENTS = True
COMMENT_VIDEOS_PER_CATEGORY = 15
MAX_COMMENTS_PER_VIDEO = 10


In [2]:
YOUTUBE_API_BASE = "https://www.googleapis.com/youtube/v3"

def youtube_get(endpoint: str, params: dict, api_key: str) -> dict:
    url = f"{YOUTUBE_API_BASE}/{endpoint}"
    payload = dict(params)
    payload["key"] = api_key
    response = requests.get(url, params=payload, timeout=30)
    if response.status_code != 200:
        safe_payload = {k: v for k, v in payload.items() if k != "key"}
        raise RuntimeError(
            f"API request failed. endpoint={endpoint}, status={response.status_code}, "
            f"params={safe_payload}, text={response.text[:250]}"
        )
    return response.json()

def fetch_all_pages(endpoint: str, params: dict, api_key: str, max_items: int | None = None) -> list:
    all_items = []
    next_page_token = None
    while True:
        request_params = dict(params)
        if next_page_token:
            request_params["pageToken"] = next_page_token
        data = youtube_get(endpoint, request_params, api_key)
        items = data.get("items", [])
        all_items.extend(items)
        if max_items is not None and len(all_items) >= max_items:
            return all_items[:max_items]
        next_page_token = data.get("nextPageToken")
        if not next_page_token:
            return all_items

def batched(values, batch_size):
    for start in range(0, len(values), batch_size):
        yield values[start:start + batch_size]

def resolve_channel(identifier: str, api_key: str) -> dict:
    if identifier.startswith("UC"):
        data = youtube_get("channels", {"part": "snippet,contentDetails", "id": identifier}, api_key)
    else:
        handle = identifier if identifier.startswith("@") else f"@{identifier}"
        data = youtube_get("channels", {"part": "snippet,contentDetails", "forHandle": handle}, api_key)
    items = data.get("items", [])
    if not items:
        raise ValueError(f"Could not resolve channel identifier: {identifier}")
    item = items[0]
    return {
        "seed_input": identifier,
        "channelId": item.get("id"),
        "channelTitle": item.get("snippet", {}).get("title"),
        "uploadsPlaylistId": item.get("contentDetails", {}).get("relatedPlaylists", {}).get("uploads"),
    }

def fetch_upload_video_ids(uploads_playlist_id: str, api_key: str, max_uploads: int) -> list:
    items = fetch_all_pages(
        "playlistItems",
        {"part": "contentDetails,snippet", "playlistId": uploads_playlist_id, "maxResults": 50},
        api_key,
        max_items=max_uploads,
    )
    rows = []
    for item in items:
        rows.append(
            {
                "videoId": item.get("contentDetails", {}).get("videoId"),
                "playlistPublishedAt": item.get("contentDetails", {}).get("videoPublishedAt"),
                "playlistPosition": item.get("snippet", {}).get("position"),
            }
        )
    return rows

def fetch_videos_by_ids(video_ids: list[str], api_key: str) -> list:
    records = []
    for id_batch in batched(video_ids, 50):
        data = youtube_get(
            "videos",
            {"part": "snippet,contentDetails,statistics", "id": ",".join(id_batch), "maxResults": 50},
            api_key,
        )
        records.extend(data.get("items", []))
    return records

def fetch_video_categories(region_code: str, api_key: str) -> pd.DataFrame:
    data = youtube_get("videoCategories", {"part": "snippet", "regionCode": region_code}, api_key)
    rows = []
    for item in data.get("items", []):
        rows.append(
            {
                "categoryId": str(item.get("id")),
                "categoryTitle": item.get("snippet", {}).get("title"),
                "categoryAssignable": item.get("snippet", {}).get("assignable"),
            }
        )
    return pd.DataFrame(rows)

def fetch_comments_for_video(video_id: str, api_key: str, max_comments: int) -> list:
    try:
        items = fetch_all_pages(
            "commentThreads",
            {
                "part": "snippet",
                "videoId": video_id,
                "maxResults": 100,
                "textFormat": "plainText",
                "order": "relevance",
            },
            api_key,
            max_items=max_comments,
        )
    except Exception:
        return []
    rows = []
    for item in items:
        top = item.get("snippet", {}).get("topLevelComment", {}).get("snippet", {})
        rows.append(
            {
                "videoId": video_id,
                "commentId": item.get("id"),
                "commentText": top.get("textDisplay") or top.get("textOriginal"),
                "commentLikeCount": top.get("likeCount"),
                "commentPublishedAt": top.get("publishedAt"),
            }
        )
    return rows

_duration_re = re.compile(r"^PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?$")

def iso_to_seconds(value: str):
    if not isinstance(value, str):
        return np.nan
    match = _duration_re.match(value)
    if not match:
        return np.nan
    hours = int(match.group(1) or 0)
    minutes = int(match.group(2) or 0)
    seconds = int(match.group(3) or 0)
    return 3600 * hours + 60 * minutes + seconds

def build_sql_database(api_key: str):
    channels_records = []
    playlist_rows = []
    for identifier in SEED_CHANNELS:
        channel_info = resolve_channel(identifier, api_key)
        channels_records.append(channel_info)
        upload_rows = fetch_upload_video_ids(channel_info["uploadsPlaylistId"], api_key, MAX_UPLOADS_PER_CHANNEL)
        for row in upload_rows:
            row["seed_input"] = channel_info["seed_input"]
            row["seed_channelId"] = channel_info["channelId"]
            row["seed_channelTitle"] = channel_info["channelTitle"]
        playlist_rows.extend(upload_rows)

    channels_df = pd.DataFrame(channels_records).drop_duplicates(subset=["channelId"]).reset_index(drop=True)
    playlist_df = pd.DataFrame(playlist_rows).dropna(subset=["videoId"]).drop_duplicates(subset=["videoId"]).reset_index(drop=True)

    video_items = fetch_videos_by_ids(playlist_df["videoId"].tolist(), api_key)
    videos_df = pd.json_normalize(video_items).rename(
        columns={
            "id": "videoId",
            "snippet.channelId": "channelId",
            "snippet.channelTitle": "channelTitle",
            "snippet.title": "title",
            "snippet.publishedAt": "publishedAt",
            "snippet.categoryId": "categoryId",
            "contentDetails.duration": "duration_iso",
            "statistics.viewCount": "viewCount",
            "statistics.likeCount": "likeCount",
            "statistics.commentCount": "commentCount",
        }
    )

    keep_cols = [
        "videoId",
        "channelId",
        "channelTitle",
        "title",
        "publishedAt",
        "categoryId",
        "duration_iso",
        "viewCount",
        "likeCount",
        "commentCount",
    ]
    videos_df = videos_df[[col for col in keep_cols if col in videos_df.columns]].copy()
    videos_df = videos_df.merge(playlist_df, on="videoId", how="left")

    categories_df = fetch_video_categories(REGION_CODE, api_key)
    category_map = dict(zip(categories_df["categoryId"], categories_df["categoryTitle"]))

    videos_df["categoryId"] = videos_df["categoryId"].astype(str)
    videos_df["categoryTitle"] = videos_df["categoryId"].map(category_map).fillna("Unknown")
    videos_df["publishedAt"] = pd.to_datetime(videos_df["publishedAt"], errors="coerce", utc=True)

    for col in ["viewCount", "likeCount", "commentCount"]:
        if col not in videos_df.columns:
            videos_df[col] = np.nan
        videos_df[col] = pd.to_numeric(videos_df[col], errors="coerce")

    videos_df["duration_seconds"] = videos_df["duration_iso"].apply(iso_to_seconds)
    now_utc = pd.Timestamp.now(tz="UTC")
    videos_df["days_since_publish"] = (
        (now_utc - videos_df["publishedAt"]).dt.total_seconds() / (24 * 3600)
    ).clip(lower=0)
    videos_df["views_per_day"] = videos_df["viewCount"] / videos_df["days_since_publish"].replace({0: np.nan})
    videos_df["like_rate"] = videos_df["likeCount"] / videos_df["viewCount"].replace({0: np.nan})
    videos_df["comment_rate"] = videos_df["commentCount"] / videos_df["viewCount"].replace({0: np.nan})
    videos_df["publish_hour_utc"] = videos_df["publishedAt"].dt.hour
    videos_df["publish_weekday"] = videos_df["publishedAt"].dt.day_name()
    videos_df["duration_bin"] = pd.cut(
        videos_df["duration_seconds"],
        bins=[-np.inf, 240, 900, np.inf],
        labels=["short (<=4m)", "medium (4m-15m)", "long (>15m)"],
    ).astype(str)

    videos_df = (
        videos_df.sort_values(["categoryTitle", "publishedAt", "videoId"], ascending=[True, False, True])
        .groupby("categoryTitle", group_keys=False)
        .apply(lambda group: group.head(min(len(group), MAX_VIDEOS_PER_CATEGORY)))
        .reset_index(drop=True)
    )

    comment_rows = []
    if FETCH_COMMENTS:
        comment_seed_ids = (
            videos_df.sort_values(["categoryTitle", "publishedAt"], ascending=[True, False])
            .groupby("categoryTitle", group_keys=False)
            .head(COMMENT_VIDEOS_PER_CATEGORY)["videoId"]
            .tolist()
        )
        for video_id in comment_seed_ids:
            comment_rows.extend(fetch_comments_for_video(video_id, api_key, MAX_COMMENTS_PER_VIDEO))

    comments_df = pd.DataFrame(comment_rows)
    if comments_df.empty:
        comments_df = pd.DataFrame(
            columns=["videoId", "commentId", "commentText", "commentLikeCount", "commentPublishedAt"]
        )
    if len(comments_df) > 0:
        comments_df["commentText"] = comments_df["commentText"].fillna("")
        comments_df["commentLength"] = comments_df["commentText"].map(len)
        comments_df["commentHasQuestion"] = comments_df["commentText"].map(lambda text: int("?" in text))
        comments_df["commentLikeCount"] = pd.to_numeric(comments_df["commentLikeCount"], errors="coerce").fillna(0)
        comment_video_features_df = (
            comments_df.groupby("videoId")
            .agg(
                sampled_comment_count=("commentId", "count"),
                avg_comment_length=("commentLength", "mean"),
                median_comment_length=("commentLength", "median"),
                share_question_comments=("commentHasQuestion", "mean"),
                avg_comment_like_count=("commentLikeCount", "mean"),
            )
            .reset_index()
        )
    else:
        comment_video_features_df = pd.DataFrame(
            columns=[
                "videoId",
                "sampled_comment_count",
                "avg_comment_length",
                "median_comment_length",
                "share_question_comments",
                "avg_comment_like_count",
            ]
        )

    if DB_PATH.exists():
        DB_PATH.unlink()

    conn = sqlite3.connect(DB_PATH)
    channels_df.to_sql("channels", conn, index=False, if_exists="replace")
    categories_df.to_sql("categories", conn, index=False, if_exists="replace")
    videos_df.assign(publishedAt=videos_df["publishedAt"].astype(str)).to_sql("videos", conn, index=False, if_exists="replace")
    comments_df.to_sql("comments", conn, index=False, if_exists="replace")
    comment_video_features_df.to_sql("comment_video_features", conn, index=False, if_exists="replace")
    return conn

conn = build_sql_database(API_KEY)
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", conn)


C:\Users\白雪琦\AppData\Local\Temp\ipykernel_7068\1273795596.py:216: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.head(min(len(group), MAX_VIDEOS_PER_CATEGORY)))


,name
0,categories
1,channels
2,comment_video_features
3,comments
4,videos


## Section 1 — Schema checks and data validation

These opening queries confirm that the relational structure needed for the project is present and that the core tables have the expected identifiers and coverage before deeper analysis begins.

## Query 1 — Table inventory and schema availability

This opening check confirms that the expected relational tables are available in the SQLite database. It is intentionally simple, because the first thing a grader or analyst should verify is whether the schema required by the project actually exists.

In [3]:
query = '''
-- Query 1: Table inventory and schema availability
-- What it does: lists the tables currently available in the SQLite database.
-- How it works: reads SQLite's sqlite_master catalog, filters to objects of type table, and orders the table names alphabetically.
-- Why it matters: this is the first sanity check that the project has the expected relational structure before any analytical SQL is run.
-- Expected output: a short table with one row per table, including the main analytical tables such as videos, channels, categories, comments, and comment_video_features.

SELECT
    name AS table_name,
    type AS object_type
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
'''
result = pd.read_sql_query(query, conn)
display(result)

,table_name,object_type
0,categories,table
1,channels,table
2,comment_video_features,table
3,comments,table
4,videos,table


## Query 2 — Row counts and key-field coverage by table

After confirming that the tables exist, this query checks whether the major tables actually contain rows and whether the key identifier fields look usable. This is a practical validation step before doing joins, rankings, and benchmark comparisons.

In [4]:
query = '''
-- Query 2: Row counts and key-field coverage by table
-- What it does: reports row counts, distinct identifier counts, and null identifier counts for the major project tables.
-- How it works: uses UNION ALL to stack compact summaries from each table so that all core tables can be validated in one scan-friendly output.
-- Why it matters: the later join, window, and subquery results are only trustworthy if the underlying tables are populated and the main identifiers are mostly unique and non-null.
-- Expected output: one row per table showing total rows, distinct key-like values, and the number of missing identifiers.

SELECT
    'videos' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT videoId) AS distinct_id_count,
    SUM(CASE WHEN videoId IS NULL THEN 1 ELSE 0 END) AS null_id_count
FROM videos

UNION ALL

SELECT
    'channels' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT channelId) AS distinct_id_count,
    SUM(CASE WHEN channelId IS NULL THEN 1 ELSE 0 END) AS null_id_count
FROM channels

UNION ALL

SELECT
    'categories' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT categoryId) AS distinct_id_count,
    SUM(CASE WHEN categoryId IS NULL THEN 1 ELSE 0 END) AS null_id_count
FROM categories

UNION ALL

SELECT
    'comments' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT commentId) AS distinct_id_count,
    SUM(CASE WHEN commentId IS NULL THEN 1 ELSE 0 END) AS null_id_count
FROM comments

UNION ALL

SELECT
    'comment_video_features' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT videoId) AS distinct_id_count,
    SUM(CASE WHEN videoId IS NULL THEN 1 ELSE 0 END) AS null_id_count
FROM comment_video_features

ORDER BY row_count DESC, table_name;
'''
result = pd.read_sql_query(query, conn)
display(result)

,table_name,row_count,distinct_id_count,null_id_count
0,comments,795,795,0
1,videos,360,360,0
2,comment_video_features,88,88,0
3,categories,32,32,0
4,channels,6,6,0


## Query 3 — Category dashboard with grouped performance summaries

This is the first true analytical query. It provides a compact dashboard of category size, growth-adjusted reach, and engagement so that the reader can quickly see how the final sample is distributed and which categories appear strongest on average.

In [5]:
query = '''
-- Query 3: Category dashboard with grouped performance summaries
-- What it does: summarizes the final video table by category and reports category-level counts plus average reach and engagement metrics.
-- How it works: groups the videos table by categoryTitle and calculates counts and category averages for views_per_day, like_rate, and comment_rate.
-- Why it matters: this is the cleanest first-pass category scorecard and supports claims in the final write-up about category composition and broad performance differences.
-- Expected output: one row per category with the number of sampled videos and several average performance metrics.

SELECT
    v.categoryTitle,
    COUNT(*) AS n_videos,
    ROUND(AVG(v.views_per_day), 2) AS avg_views_per_day,
    ROUND(AVG(v.like_rate), 4) AS avg_like_rate,
    ROUND(AVG(v.comment_rate), 4) AS avg_comment_rate
FROM videos v
GROUP BY v.categoryTitle
ORDER BY n_videos DESC, avg_views_per_day DESC;
'''
result = pd.read_sql_query(query, conn)
display(result.head(50))

,categoryTitle,n_videos,avg_views_per_day,avg_like_rate,avg_comment_rate
0,News & Politics,60,640233.37,0.0240,0.0063
1,Sports,60,163697.28,0.0266,0.0018
2,Science & Technology,60,157549.32,0.0292,0.0021
3,Entertainment,60,19033.06,0.0297,0.0020
4,Music,60,12780.24,0.0317,0.0015
5,People & Blogs,60,3936.76,0.0225,0.0010


## Query 4 — Join videos to categories to verify readable category mapping

This query keeps the analysis close to the relational design. It shows that the category lookup table connects correctly to the main video table and produces an audit-friendly sample with both numeric and readable category fields.

In [6]:
query = '''
-- Query 4: Video-to-category join for readable category mapping
-- What it does: joins the final videos table to the categories lookup table and displays sample video records with both category IDs and category titles.
-- How it works: performs a LEFT JOIN from videos to categories on categoryId, retains row-level video detail, and orders the result by views_per_day.
-- Why it matters: this verifies that the category lookup table is connected correctly and shows the grader that the relational mapping behind later category analysis is working as intended.
-- Expected output: a sample of videos with titles, numeric category IDs, readable category names, and a growth-adjusted reach metric.

SELECT
    v.videoId,
    v.title,
    v.categoryId,
    c.categoryTitle,
    ROUND(v.views_per_day, 2) AS views_per_day
FROM videos v
LEFT JOIN categories c
    ON v.categoryId = c.categoryId
ORDER BY v.views_per_day DESC
LIMIT 25;
'''
result = pd.read_sql_query(query, conn)
display(result.head(50))

,videoId,title,categoryId,categoryTitle,views_per_day
0,ItpjHZYdv_M,Watters: Fear is setting in...,25,News & Politics,4284371.65
1,wbAAIfHjtBA,BREAKING: Israel UNLEASHES massive strikes on ...,25,News & Politics,3223416.69
2,6eP7mb89-7E,JUST IN: Israeli strikes hit Beirut,25,News & Politics,3059476.10
3,YiJytDvKx_4,US airpower about to 'SURGE DRAMATICALLY' over...,25,News & Politics,2707732.19
4,kBX5WH9b4M4,Macbook Neo Impressions: Reincarnated!,28,Science & Technology,2641660.09
5,nfHRMqqO578,Samsung Galaxy S26 Ultra Review: There's a Catch,28,Science & Technology,2474842.93
6,efuQvGsU1yA,JUST IN: Jets STRIKE Khamenei's bunker,25,News & Politics,2224738.18
7,U1tUvYHqgFM,MAVERICKS at CELTICS | FULL GAME HIGHLIGHTS | ...,17,Sports,1754863.17
8,i7X3tAhFB4g,Kurdish leader addresses potential military op...,25,News & Politics,1531043.88
9,b1eFpPixR8Q,JUST IN: Russia aiding Iran in targeting US as...,25,News & Politics,1396537.71


## Query 5 — Join videos to channels to compare channel-category combinations

The next step is to move from category-level summaries to channel-category combinations. This helps identify whether certain channels appear especially strong within specific content types.

In [7]:
query = '''
-- Query 5: Channel-category performance via a videos-to-channels join
-- What it does: aggregates the video table by channel and category to compare how different channel-category combinations perform.
-- How it works: INNER JOINs videos to channels on channelId, groups by channelTitle and categoryTitle, filters to combinations with at least three videos, and computes average reach and like-rate metrics.
-- Why it matters: this supports channel-level strategy questions by showing whether performance differences are only about categories or also about which channels compete inside them.
-- Expected output: one row per channel-category combination with counts and average performance measures.

SELECT
    ch.channelTitle,
    v.categoryTitle,
    COUNT(*) AS n_videos,
    ROUND(AVG(v.views_per_day), 2) AS avg_views_per_day,
    ROUND(AVG(v.like_rate), 4) AS avg_like_rate
FROM videos v
INNER JOIN channels ch
    ON v.channelId = ch.channelId
GROUP BY ch.channelTitle, v.categoryTitle
HAVING COUNT(*) >= 3
ORDER BY avg_views_per_day DESC, avg_like_rate DESC;
'''
result = pd.read_sql_query(query, conn)
display(result.head(50))

,channelTitle,categoryTitle,n_videos,avg_views_per_day,avg_like_rate
0,Fox News,News & Politics,60,640233.37,0.0240
1,Marques Brownlee,Science & Technology,27,347619.85,0.0347
2,NBA,Sports,60,163697.28,0.0266
3,WIRED,Entertainment,59,19348.68,0.0298
4,NPR Music,Music,60,12780.24,0.0317
5,TED,People & Blogs,60,3936.76,0.0225
6,TED,Science & Technology,33,2037.07,0.0247


## Query 6 — Join comment features to videos to connect audience response with category outcomes

This query brings in the second dataset derived from comments. It asks whether categories with richer audience response also look stronger on standard engagement metrics.

In [8]:
query = '''
-- Query 6: Comment features joined back to category outcomes
-- What it does: joins video-level comment features to the video table and summarizes comment intensity and engagement by category.
-- How it works: INNER JOINs videos to comment_video_features on videoId, groups the matched rows by categoryTitle, and computes averages for comment length, question-comment share, like_rate, and comment_rate.
-- Why it matters: this moves the project beyond exposure alone and supports write-up claims about audience response quality, not just audience size.
-- Expected output: one row per category showing how comment-based features and engagement rates line up when the second dataset is merged back in.

SELECT
    v.categoryTitle,
    COUNT(*) AS n_videos_with_comment_sample,
    ROUND(AVG(cvf.avg_comment_length), 2) AS avg_comment_length,
    ROUND(AVG(cvf.share_question_comments), 4) AS avg_share_question_comments,
    ROUND(AVG(v.like_rate), 4) AS avg_like_rate,
    ROUND(AVG(v.comment_rate), 4) AS avg_comment_rate
FROM videos v
INNER JOIN comment_video_features cvf
    ON v.videoId = cvf.videoId
GROUP BY v.categoryTitle
ORDER BY avg_comment_length DESC, avg_like_rate DESC;
'''
result = pd.read_sql_query(query, conn)
display(result.head(50))

,categoryTitle,n_videos_with_comment_sample,avg_comment_length,avg_share_question_comments,avg_like_rate,avg_comment_rate
0,Entertainment,15,110.42,0.1233,0.0294,0.0015
1,People & Blogs,14,95.16,0.1421,0.0173,0.0011
2,Science & Technology,15,90.77,0.1600,0.0333,0.0023
3,Music,14,83.09,0.0143,0.0426,0.0024
4,News & Politics,15,66.41,0.1467,0.0378,0.0078
5,Sports,15,49.76,0.0179,0.0403,0.0033


## Query 7 — Grouped summary by weekday and duration bin

This grouped comparison is operational rather than purely descriptive. It helps the target audience think jointly about scheduling decisions and content format choices.

In [9]:
query = '''
-- Query 7: Weekday-by-duration grouped comparison
-- What it does: summarizes video counts and average performance metrics for each publish_weekday and duration_bin combination.
-- How it works: groups the videos table by publish_weekday and duration_bin, filters out extremely sparse cells with HAVING COUNT(*) >= 2, and reports average growth-adjusted reach and like rates.
-- Why it matters: the result is useful for practical publishing decisions because managers often need to think about timing and format together rather than one at a time.
-- Expected output: one row per weekday-duration combination with the number of videos and average performance metrics for that cell.

SELECT
    publish_weekday,
    duration_bin,
    COUNT(*) AS n_videos,
    ROUND(AVG(views_per_day), 2) AS avg_views_per_day,
    ROUND(AVG(like_rate), 4) AS avg_like_rate
FROM videos
GROUP BY publish_weekday, duration_bin
HAVING COUNT(*) >= 2
ORDER BY publish_weekday, duration_bin;
'''
result = pd.read_sql_query(query, conn)
display(result.head(50))

,publish_weekday,duration_bin,n_videos,avg_views_per_day,avg_like_rate
0,Friday,long (>15m),36,126998.16,0.0245
1,Friday,medium (4m-15m),39,590133.69,0.0233
2,Friday,short (<=4m),57,151719.04,0.0236
3,Monday,long (>15m),11,33385.96,0.0365
4,Monday,medium (4m-15m),13,4386.23,0.0263
5,Monday,short (<=4m),4,1017.68,0.0185
6,Saturday,long (>15m),8,353384.85,0.0285
7,Saturday,medium (4m-15m),10,875782.85,0.0467
8,Saturday,short (<=4m),24,125690.17,0.0374
9,Sunday,long (>15m),2,1757.19,0.0244


## Query 8 — Three-table join from comments to videos to categories

The previous comment query used pre-aggregated comment features. This query goes back to the raw comments table and joins through the relational structure to summarize a concrete comment behavior by category.

In [10]:
query = '''
-- Query 8: Raw comments joined through videos to categories
-- What it does: joins raw comments to videos and categories, then summarizes comment volume, question-mark usage, and comment length by category.
-- How it works: starts from the comments table, INNER JOINs to videos on videoId and to categories on categoryId, then aggregates comment-level text features by category.
-- Why it matters: this shows that the project can analyze raw audience text directly and not only rely on already-aggregated comment features.
-- Expected output: one row per category with sampled comment counts and category-level summaries of comment style.

SELECT
    c.categoryTitle,
    COUNT(cm.commentId) AS sampled_comments,
    ROUND(AVG(CASE WHEN INSTR(cm.commentText, '?') > 0 THEN 1.0 ELSE 0.0 END), 4) AS share_comments_with_question,
    ROUND(AVG(LENGTH(cm.commentText)), 2) AS avg_comment_length
FROM comments cm
INNER JOIN videos v
    ON cm.videoId = v.videoId
INNER JOIN categories c
    ON v.categoryId = c.categoryId
GROUP BY c.categoryTitle
ORDER BY share_comments_with_question DESC, avg_comment_length DESC;
'''
result = pd.read_sql_query(query, conn)
display(result.head(50))

,categoryTitle,sampled_comments,share_comments_with_question,avg_comment_length
0,Science & Technology,150,0.1600,90.77
1,People & Blogs,109,0.1468,101.87
2,News & Politics,150,0.1467,66.41
3,Entertainment,148,0.1216,111.41
4,Music,114,0.0175,93.91
5,Sports,124,0.0161,52.02


## Query 9 — Window function to rank videos within each category

This is the first window-function query. Instead of collapsing the data, it keeps row-level detail while still creating a fair within-category leaderboard.

In [11]:
query = '''
-- Query 9: Within-category ranking of videos by views per day
-- What it does: ranks videos by views_per_day inside each category while keeping each video as its own row.
-- How it works: uses ROW_NUMBER() and RANK() window functions partitioned by categoryTitle and ordered by views_per_day in descending order.
-- Why it matters: this lets the project identify category leaders without unfairly comparing large and small categories on one pooled scale.
-- Expected output: row-level video observations with within-category rank numbers attached.

SELECT
    v.categoryTitle,
    v.title,
    ROUND(v.views_per_day, 2) AS views_per_day,
    ROW_NUMBER() OVER (
        PARTITION BY v.categoryTitle
        ORDER BY v.views_per_day DESC
    ) AS row_number_in_category,
    RANK() OVER (
        PARTITION BY v.categoryTitle
        ORDER BY v.views_per_day DESC
    ) AS rank_in_category
FROM videos v
WHERE v.views_per_day IS NOT NULL
ORDER BY v.categoryTitle, rank_in_category
LIMIT 40;
'''
result = pd.read_sql_query(query, conn)
display(result.head(50))

,categoryTitle,title,views_per_day,row_number_in_category,rank_in_category
0,Entertainment,F1 Chief Mechanic Answers F1 Car Questions | T...,386217.96,1,1
1,Entertainment,How is Temu so cheap?,214415.78,2,2
2,Entertainment,Your Rich BFF Vivian Tu Answers Personal Finan...,52476.46,3,3
3,Entertainment,Finance Professor Answers Investing Questions ...,38810.14,4,4
4,Entertainment,Voice Acting Legend Jim Cummings Answers Voice...,33044.27,5,5
5,Entertainment,Stranger Things Cast Answer The 50 Most Search...,29652.02,6,6
6,Entertainment,MrBeast Answers The Web's Most Searched Questi...,26741.57,7,7
7,Entertainment,Alex Honnold Answers Rock Climbing Questions |...,23260.16,8,8
8,Entertainment,Jamie Campbell Bower Answers The Web's Most Se...,23224.39,9,9
9,Entertainment,Supply Chain Expert Answers Chinese Manufactur...,22929.51,10,10


## Query 10 — Window function to measure concentration within category

Once the top videos are identified, the next question is whether category performance is broad-based or dominated by a few breakout uploads. This query uses cumulative shares to make that visible.

In [12]:
query = '''
-- Query 10: Within-category concentration of total views
-- What it does: computes cumulative views and cumulative view share within each category after sorting videos from highest to lowest viewCount.
-- How it works: uses SUM() OVER window functions partitioned by categoryTitle and ordered by descending viewCount to build running totals and running shares.
-- Why it matters: this supports claims about concentration by showing whether a category's total reach is spread across many videos or heavily driven by a small number of leaders.
-- Expected output: row-level videos ordered within category with cumulative counts and cumulative share values that rise toward one.

SELECT
    categoryTitle,
    title,
    viewCount,
    SUM(viewCount) OVER (
        PARTITION BY categoryTitle
        ORDER BY viewCount DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_views_in_category,
    ROUND(
        1.0 * SUM(viewCount) OVER (
            PARTITION BY categoryTitle
            ORDER BY viewCount DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
        / NULLIF(SUM(viewCount) OVER (PARTITION BY categoryTitle), 0),
        4
    ) AS cumulative_view_share_in_category
FROM videos
ORDER BY categoryTitle, viewCount DESC
LIMIT 50;
'''
result = pd.read_sql_query(query, conn)
display(result.head(50))

,categoryTitle,title,viewCount,cumulative_views_in_category,cumulative_view_share_in_category
0,Entertainment,Stranger Things Cast Answer The 50 Most Search...,2978612,2978612,0.0859
1,Entertainment,Historian Answers Folklore Questions | Tech Su...,2690656,5669268,0.1635
2,Entertainment,Alex Honnold Answers Rock Climbing Questions |...,2380151,8049419,0.2321
3,Entertainment,Gordon Ramsay Answers The Web's Most Searched ...,1874321,9923740,0.2862
4,Entertainment,How is Temu so cheap?,1797758,11721498,0.3380
5,Entertainment,Jamie Campbell Bower Answers The Web's Most Se...,1729108,13450606,0.3879
6,Entertainment,Hideo Kojima Answers Hideo Kojima Questions | ...,1686606,15137212,0.4365
7,Entertainment,MrBeast Answers The Web's Most Searched Questi...,1372574,16509786,0.4761
8,Entertainment,Army Historian Answers World War II Questions ...,1063890,17573676,0.5068
9,Entertainment,Professor Answers Coding Questions | Tech Supp...,1050893,18624569,0.5371


## Query 11 — Window function for the latest videos by channel

This final window query gives a managerial recency view. It helps the reader inspect current publishing behavior within each channel rather than only historical averages.

In [13]:
query = '''
-- Query 11: Latest videos within each channel
-- What it does: assigns a recency rank to each video inside its channel based on publishedAt.
-- How it works: applies ROW_NUMBER() as a window function partitioned by channelTitle and ordered by publishedAt descending so newer uploads receive smaller rank numbers.
-- Why it matters: this gives the write-up a way to comment on current publishing patterns and whether recent uploads line up with broader channel-category trends.
-- Expected output: row-level video data with a recency rank inside each channel.

SELECT
    channelTitle,
    categoryTitle,
    title,
    publishedAt,
    ROW_NUMBER() OVER (
        PARTITION BY channelTitle
        ORDER BY publishedAt DESC
    ) AS recency_rank_in_channel
FROM videos
ORDER BY channelTitle, recency_rank_in_channel
LIMIT 40;
'''
result = pd.read_sql_query(query, conn)
display(result.head(50))

,channelTitle,categoryTitle,title,publishedAt,recency_rank_in_channel
0,Fox News,News & Politics,Sen Kennedy says Trump was 'as mad as a mama w...,2026-03-07 03:30:16+00:00,1
1,Fox News,News & Politics,"Watters: Iran is bruised, battered and confused",2026-03-07 03:00:00+00:00,2
2,Fox News,News & Politics,‘The Five’: Dems might regret this…,2026-03-07 02:45:02+00:00,3
3,Fox News,News & Politics,Morale could be cratering among Iranian milita...,2026-03-07 02:00:30+00:00,4
4,Fox News,News & Politics,Watters: Fear is setting in...,2026-03-07 02:00:06+00:00,5
5,Fox News,News & Politics,Kurdish leader addresses potential military op...,2026-03-07 01:15:04+00:00,6
6,Fox News,News & Politics,POULTRY PURSUIT: Officers in Georgia respond t...,2026-03-07 01:00:14+00:00,7
7,Fox News,News & Politics,‘The Five’: Deal or no deal?,2026-03-07 00:30:06+00:00,8
8,Fox News,News & Politics,RAGING INFERNO: Firefighters battle a five-ala...,2026-03-07 00:00:55+00:00,9
9,Fox News,News & Politics,US forces deliver MAJOR blows to Iran's navy,2026-03-06 23:45:00+00:00,10


## Query 12 — Correlated subquery for above-category-average videos

The first subquery-based benchmark asks which videos outperform the average video in their own category. This is a fairer benchmark than comparing every video to one overall average.

In [14]:
query = '''
-- Query 12: Videos above their own category average
-- What it does: keeps only videos whose views_per_day exceed the average views_per_day for their own category.
-- How it works: uses a correlated subquery in the WHERE clause so each video's benchmark is the average of the rows that share its categoryTitle.
-- Why it matters: this identifies category-relative outperformers and supports claims about within-category leaders that are not driven by category size differences.
-- Expected output: a filtered list of videos that beat the average performance level inside their own category.

SELECT
    v.videoId,
    v.categoryTitle,
    v.title,
    ROUND(v.views_per_day, 2) AS views_per_day
FROM videos v
WHERE v.views_per_day > (
    SELECT AVG(v2.views_per_day)
    FROM videos v2
    WHERE v2.categoryTitle = v.categoryTitle
)
ORDER BY v.categoryTitle, v.views_per_day DESC
LIMIT 40;
'''
result = pd.read_sql_query(query, conn)
display(result.head(50))

,videoId,categoryTitle,title,views_per_day
0,Ie-KCHHSUo4,Entertainment,F1 Chief Mechanic Answers F1 Car Questions | T...,386217.96
1,8QhDb1d5JN4,Entertainment,How is Temu so cheap?,214415.78
2,xbgxajnBrak,Entertainment,Your Rich BFF Vivian Tu Answers Personal Finan...,52476.46
3,k-HZIFIoy3Q,Entertainment,Finance Professor Answers Investing Questions ...,38810.14
4,Fh9sDpJSnrY,Entertainment,Voice Acting Legend Jim Cummings Answers Voice...,33044.27
5,l5Y02Yadz3A,Entertainment,Stranger Things Cast Answer The 50 Most Search...,29652.02
6,FWl-LMxx5sg,Entertainment,MrBeast Answers The Web's Most Searched Questi...,26741.57
7,yDTg4P9ZdP4,Entertainment,Alex Honnold Answers Rock Climbing Questions |...,23260.16
8,0_bkIT11zcI,Entertainment,Jamie Campbell Bower Answers The Web's Most Se...,23224.39
9,L-fK_BUmesc,Entertainment,Supply Chain Expert Answers Chinese Manufactur...,22929.51


## Query 13 — Uncorrelated subquery for categories above the overall average like rate

This query shifts the benchmark from within-category comparison to an overall sample-wide comparison. It identifies which categories sit above the global engagement average.

In [15]:
query = '''
-- Query 13: Categories above the overall average like rate
-- What it does: returns categories whose average like_rate is greater than the overall average like_rate across the full video sample.
-- How it works: first builds category-level averages in a grouped subquery, then filters those rows against an uncorrelated subquery that computes the overall benchmark.
-- Why it matters: this produces a clean above-benchmark category list that can be referenced directly in the final report when discussing relative engagement leaders.
-- Expected output: one row per above-benchmark category, sorted from the highest average like rate downward.

SELECT
    categoryTitle,
    avg_like_rate
FROM (
    SELECT
        categoryTitle,
        AVG(like_rate) AS avg_like_rate
    FROM videos
    GROUP BY categoryTitle
)
WHERE avg_like_rate > (
    SELECT AVG(like_rate)
    FROM videos
)
ORDER BY avg_like_rate DESC;
'''
result = pd.read_sql_query(query, conn)
display(result.head(50))

,categoryTitle,avg_like_rate
0,Music,0.031686
1,Entertainment,0.029741
2,Science & Technology,0.029189


## Query 14 — Subquery to find videos whose sampled comments are longer than the overall average

This last benchmark query returns to audience response quality. It identifies videos whose sampled comments are longer than the sample-wide average comment length.

In [16]:
query = '''
-- Query 14: Videos with longer-than-average sampled comments
-- What it does: keeps videos whose average sampled comment length is above the overall average comment length across all video-level comment summaries.
-- How it works: INNER JOINs videos to comment_video_features and filters the joined rows with an uncorrelated subquery that computes the global average avg_comment_length.
-- Why it matters: longer comments can be a rough signal of more involved audience response, so this query helps connect conversation depth to video outcomes.
-- Expected output: a filtered list of videos with unusually long sampled comments, along with category and engagement context.

SELECT
    v.categoryTitle,
    v.title,
    ROUND(cvf.avg_comment_length, 2) AS avg_comment_length,
    ROUND(v.like_rate, 4) AS like_rate
FROM videos v
INNER JOIN comment_video_features cvf
    ON v.videoId = cvf.videoId
WHERE cvf.avg_comment_length > (
    SELECT AVG(avg_comment_length)
    FROM comment_video_features
)
ORDER BY cvf.avg_comment_length DESC, v.like_rate DESC
LIMIT 40;
'''
result = pd.read_sql_query(query, conn)
display(result.head(50))

,categoryTitle,title,avg_comment_length,like_rate
0,Music,Ganavya: Tiny Desk Concert,297.30,0.0519
1,Entertainment,Professor Answers Olympic History Questions | ...,232.80,0.0224
2,People & Blogs,What Ancestral Intelligence Can Teach Us About...,206.70,0.0254
3,Science & Technology,My Year Living with a Robot | Emily Kate Genat...,186.00,0.0305
4,People & Blogs,The Case for Spending More Time with Your Frie...,185.00,0.0208
5,Music,Sarah McLachlan: Tiny Desk Concert,167.50,0.0346
6,Science & Technology,How to Power the World 24/7 — Without Oil | Ci...,154.00,0.0222
7,Entertainment,Your Rich BFF Vivian Tu Answers Personal Finan...,143.90,0.0397
8,News & Politics,Kurdish leader addresses potential military op...,143.80,0.0263
9,People & Blogs,Why Can’t We Better Prepare for Extreme Weathe...,142.00,0.0197


## Query 15 — Final decision-oriented category scorecard

The notebook closes with one integrated scorecard that pulls together category size, reach, engagement, and audience-response measures. This is the most presentation-ready SQL output in the notebook and is the easiest table to reference in the final write-up.

In [17]:
query = '''
-- Query 15: Final decision-oriented category scorecard
-- What it does: combines category size, reach, engagement, and comment-based response metrics into one presentation-ready category table.
-- How it works: LEFT JOINs videos to comment_video_features on videoId, groups by categoryTitle, and computes averages so that all major category-level metrics appear in one output.
-- Why it matters: this is the most decision-ready summary in the notebook because it pulls the main project dimensions into one place for the final write-up or presentation slides.
-- Expected output: one row per category with counts plus category-level measures of reach, engagement, and audience-response quality.

SELECT
    v.categoryTitle,
    COUNT(*) AS n_videos,
    ROUND(AVG(v.views_per_day), 2) AS avg_views_per_day,
    ROUND(AVG(v.like_rate), 4) AS avg_like_rate,
    ROUND(AVG(v.comment_rate), 4) AS avg_comment_rate,
    ROUND(AVG(cvf.avg_comment_length), 2) AS avg_comment_length,
    ROUND(AVG(cvf.share_question_comments), 4) AS avg_share_question_comments
FROM videos v
LEFT JOIN comment_video_features cvf
    ON v.videoId = cvf.videoId
GROUP BY v.categoryTitle
ORDER BY avg_views_per_day DESC, avg_like_rate DESC, avg_comment_rate DESC;
'''
result = pd.read_sql_query(query, conn)
display(result.head(50))

,categoryTitle,n_videos,avg_views_per_day,avg_like_rate,avg_comment_rate,avg_comment_length,avg_share_question_comments
0,News & Politics,60,640233.37,0.0240,0.0063,66.41,0.1467
1,Sports,60,163697.28,0.0266,0.0018,49.76,0.0179
2,Science & Technology,60,157549.32,0.0292,0.0021,90.77,0.1600
3,Entertainment,60,19033.06,0.0297,0.0020,110.42,0.1233
4,Music,60,12780.24,0.0317,0.0015,83.09,0.0143
5,People & Blogs,60,3936.76,0.0225,0.0010,95.16,0.1421


## How these SQL results connect to the final write-up

The most direct tables to reference in the final Python/write-up notebook are:

- **Query 3** for category composition and broad performance differences.
- **Query 6** and **Query 14** for audience-response quality and comment intensity.
- **Query 9** and **Query 12** for within-category leaders and fair benchmarking.
- **Query 10** for concentration and breakout-video dependence.
- **Query 15** for a final managerial scorecard that combines the main dimensions in one place.

This makes the SQL notebook function as an analytical backbone rather than a disconnected appendix.

## Close the connection

The sequence above now covers schema checks, validation, grouped summaries, join-based analysis, window functions, subqueries, and a final scorecard. Every query is documented in the same what / how / why / expected-output format so the SQL can be graded quickly and cited directly in the final project narrative.

In [18]:
conn.close()
